# Feature-group importance for |N error|, all models

Per model: top-10% vs bottom-50% |error| classifier (HistGradientBoosting, GroupKFold by protein) and grouped permutation importance with shared feature groups.

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score


def set_style(base=13, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base,
        "axes.titlesize": base + 2, "axes.labelsize": base, "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2, "legend.fontsize": base - 2, "svg.fonttype": "none"})


set_style()

FIG4 = "../../data/analysis/Fig4"
AF = f"{FIG4}/all_features"
GT_ROOT = "../../data/rBMRB/cluster_30p"
FIG = "Figs"

MODELS = {                # file suffix -> label; order = plotting order
    "":              "SLAE-CS",
    "_esm3":         "ESM3 (str + seq)",
    "_esm3_seqonly": "ESM3 (seq only)",
    "_mace":         "MACE-OFF23",
    "_mpnn_ft":      "ProteinMPNN (finetuned)",
}
TAG_NAME = {t: (t.lstrip("_") or "slaecs") for t in MODELS}     # used in output file names
OUT_SUFFIX = "_panav"                                            # e.g. ..._panav_slaecs.csv


PRR_SS = f"{FIG4}/sasa_results/per_residue_records.json"   # ss_class source used by the error recipe
prr_ss = json.load(open(PRR_SS))


def abserr_table(tag, gt_data):
    """Per-residue |error| for one model -- same recipe as Fig4/ss_effect_sweep_panav_10seedens.ipynb (cell 2):
    PANAV test.json, 10-seed ensemble `mean`/`mask`, Proline / None / prediction-mask gate, no analytical
    centering, padding-safe truncation, residue kept only if per_residue_records.json gives it an ss_class."""
    pred_data = json.load(open(f"{FIG4}/results/predictions_N_seed_ensemble_10seed{tag}.json"))
    pred_dict, mask_dict = pred_data['mean'], pred_data['mask']   # 10-seed ensemble mean, not per-seed 'pred'
    dictionary = {v: k for k, v in gt_data['entry_ID'].items()}

    rows = []
    for entry in gt_data['entry_ID'].values():
        key = dictionary.get(entry)
        if key is None or entry not in pred_dict:
            continue
        prr_entry = prr_ss.get(entry)
        if prr_entry is None:
            continue

        gt_seq, gt_shifts = gt_data['sequence'][key], gt_data['N'][key]
        pred_arr_full = np.array(pred_dict[entry])
        pred_mask_full = np.array(mask_dict[entry], dtype=bool)
        n = len(gt_seq)
        # predictions are batch-padded: truncate to n instead of dropping the protein
        if len(pred_arr_full) < n or len(pred_mask_full) < n:
            continue
        pred_arr = pred_arr_full[:n]
        pred_mask = pred_mask_full[:n]

        prr_lookup = {r['resnum']: r for r in prr_entry}

        # Proline / None exclusion + the prediction's own mask (tag/flag/outlier exclusion folded in)
        gt_mask_1 = np.array([res != 'P' for res in gt_seq])
        none_mask = np.array([g is not None for g in gt_shifts])
        mask = gt_mask_1 & none_mask & pred_mask
        if mask.sum() == 0:
            continue

        full_indices = np.where(mask)[0]
        pred_f = pred_arr[mask]
        gt_f = np.array(gt_shifts)[mask].astype(float)
        abs_error = np.abs(pred_f - gt_f)          # no analytical centering (PANAV referencing already consistent)

        for i, full_idx in enumerate(full_indices):
            resnum = int(full_idx) + 1
            rec = prr_lookup.get(resnum)
            if rec is None or rec.get('ss_class') is None:
                continue
            rows.append((entry, resnum, float(abs_error[i])))

    ae = pd.DataFrame(rows, columns=["entry", "resnum", "abserr"])
    print(f"  {MODELS[tag]:22s} rows={len(ae)} proteins={ae.entry.nunique()}")
    return ae


gt = json.load(open(f"{GT_ROOT}/PANAV/test.json"))       # PANAV referencing, same ground truth as Fig4
print("ground truth: PANAV/test.json (no per-protein offset correction)")
errs = {tag: abserr_table(tag, gt) for tag in MODELS}

# ---- per-residue features (all_features/per_residue_records_full.json), model-independent ----
GEOM_CAT = ["prev_aa", "next_aa", "hb_HN_exists", "is_prepro"]
GEOM_NUM = ["phi_sin", "phi_cos", "psi_sin", "psi_cos", "psi_prev_sin", "psi_prev_cos",
            "phi_next_sin", "phi_next_cos", "chi1_sin", "chi1_cos",
            "hb_HN_dist", "hb_HN_cosA", "hb_HN_cosH", "hb_HN_energy", "rc_N", "pack_N8"]
BASE_CAT = ["aa", "tdi", "tdn", "ss", "exposure"]
BASE_NUM = ["s2", "ph", "temperature", "p_exchange", "plddt"]

prr = json.load(open(f"{AF}/per_residue_records_full.json"))
rows = [(e, r["resnum"], r["residue_type"], r["tdi"], r["tdn"], r["ss_class"], r["exposure"],
         r["s2"], r["ph"], r["temperature"], r["p_exchange"], r["plddt"],
         *[r[c] for c in GEOM_CAT], *[r[c] for c in GEOM_NUM])
        for e, rl in prr.items() for r in rl]
feat = pd.DataFrame(rows, columns=["entry", "resnum"] + BASE_CAT + BASE_NUM + GEOM_CAT + GEOM_NUM)

# one table, one |error| column per model; residues = gated set (identical across models, checked here)
keysets = {tag: set(zip(ae.entry, ae.resnum)) for tag, ae in errs.items()}
ref_keys = keysets[""]
for tag, ks in keysets.items():
    if ks != ref_keys:
        raise ValueError(f"{MODELS[tag]}: gated residue set differs from SLAE-CS "
                         f"(+{len(ks - ref_keys)} / -{len(ref_keys - ks)}) -- shared groups would not be exact")
df = feat.merge(errs[""].rename(columns={"abserr": "err"}), on=["entry", "resnum"], how="inner")
for tag in list(MODELS)[1:]:
    df = df.merge(errs[tag].rename(columns={"abserr": f"err{tag}"}), on=["entry", "resnum"], how="left")
for c in BASE_NUM + GEOM_NUM:
    df[c] = df[c].fillna(df[c].median())
for c in GEOM_CAT:
    df[c] = df[c].fillna("NA")
print(f"\nfeature table: rows={len(df)} proteins={df.entry.nunique()}")

In [ ]:
# ---- unified association matrix over all features (model-independent) ----
from scipy.stats import chi2_contingency, spearmanr
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import squareform
import matplotlib.patches as mpatches

CAT_FEATS = BASE_CAT + GEOM_CAT
NUM_FEATS = BASE_NUM + GEOM_NUM
FEATS = CAT_FEATS + NUM_FEATS
DA = df


def cramers_v(a, b):                                  # bias-corrected (Bergsma 2013)
    ct = pd.crosstab(a, b)
    if min(ct.shape) < 2: return 0.0
    chi2 = chi2_contingency(ct, correction=False)[0]; n = ct.values.sum(); r, k = ct.shape
    phi2c = max(0.0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1); kc = k - (k - 1) ** 2 / (n - 1); den = min(kc - 1, rc - 1)
    return float(np.sqrt(phi2c / den)) if den > 0 else 0.0


def corr_ratio(cats, vals):                           # eta (categorical -> numeric)
    cats = np.asarray(cats); vals = np.asarray(vals, float)
    m = ~np.isnan(vals); cats, vals = cats[m], vals[m]
    if len(vals) == 0: return 0.0
    g = vals.mean(); sst = ((vals - g) ** 2).sum()
    if sst == 0: return 0.0
    ssb = sum(len(vals[cats == u]) * (vals[cats == u].mean() - g) ** 2 for u in np.unique(cats))
    return float(np.sqrt(ssb / sst))


M = pd.DataFrame(np.eye(len(FEATS)), index=FEATS, columns=FEATS)
for i, a in enumerate(FEATS):
    for b in FEATS[i + 1:]:
        if a in CAT_FEATS and b in CAT_FEATS:
            v = cramers_v(DA[a], DA[b])
        elif a in NUM_FEATS and b in NUM_FEATS:
            v = abs(spearmanr(DA[a], DA[b]).correlation)
        else:
            cc, nn = (a, b) if a in CAT_FEATS else (b, a)
            v = corr_ratio(DA[cc], DA[nn])
        M.loc[a, b] = M.loc[b, a] = 0.0 if np.isnan(v) else v
print("unified association matrix:", M.shape, "| residues used:", len(DA))

In [ ]:
# ---- cluster -> empirical groups, clustered heatmap (model-independent) ----
Dist = 1 - M.values; np.fill_diagonal(Dist, 0.0); Dist = (Dist + Dist.T) / 2
Z = linkage(squareform(Dist, checks=False), method="average")
order = leaves_list(Z); FEAT_ORD = [FEATS[i] for i in order]

ASSOC_THR = 0.7                                   # association >= THR  -> same empirical group
cl = fcluster(Z, t=1 - ASSOC_THR, criterion="distance")
gmap = {}
for f, c in zip(FEATS, cl): gmap.setdefault(c, []).append(f)
emp_groups = sorted(gmap.values(), key=lambda g: -len(g))
print(f"=== empirical groups (association >= {ASSOC_THR}) ===")
for g in emp_groups:
    if len(g) > 1: print("  GROUP:", g)
print("  singletons:", [g[0] for g in emp_groups if len(g) == 1])

set_style(11)
Mo = M.loc[FEAT_ORD, FEAT_ORD]
fig, ax = plt.subplots(figsize=(14.5, 13))
im = ax.imshow(Mo.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(FEAT_ORD))); ax.set_xticklabels(FEAT_ORD, rotation=55, ha="right", fontsize=9)
ax.set_yticks(range(len(FEAT_ORD))); ax.set_yticklabels(FEAT_ORD, fontsize=9)
for i in range(len(FEAT_ORD)):
    for j in range(len(FEAT_ORD)):
        v = Mo.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9, color="white" if v > 0.5 else "black")
ord_cl = [cl[order[i]] for i in range(len(order))]
s = 0
for i in range(1, len(ord_cl) + 1):
    if i == len(ord_cl) or ord_cl[i] != ord_cl[s]:
        if i - s > 1:
            ax.add_patch(mpatches.Rectangle((s - 0.5, s - 0.5), i - s, i - s, fill=False, ec="#F32D13", lw=2.2))
        s = i
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("association   (|Spearman| / Cramér's V / η)")
plt.tight_layout()
plt.savefig(f"{FIG}/error_enrichment_feature_association_all{OUT_SUFFIX}.svg", bbox_inches="tight")
plt.savefig(f"{FIG}/error_enrichment_feature_association_all{OUT_SUFFIX}.png", dpi=300, bbox_inches="tight")
print(f"saved error_enrichment_feature_association_all{OUT_SUFFIX}"); plt.show()

In [ ]:
# ---- hybrid groups: empirical clusters + must-link (dihedral sin/cos, hb_HN_*), shared by all models ----
ATOMIC = [["phi_sin", "phi_cos"], ["psi_sin", "psi_cos"], ["psi_prev_sin", "psi_prev_cos"],
          ["phi_next_sin", "phi_next_cos"], ["chi1_sin", "chi1_cos"],
          ["hb_HN_dist", "hb_HN_cosA", "hb_HN_cosH", "hb_HN_exists", "hb_HN_energy"]]

parent = {f: f for f in FEATS}
def _find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
def _union(a, b): parent[_find(a)] = _find(b)
for g in emp_groups:
    for f in g[1:]: _union(g[0], f)
for group in ATOMIC:
    members = [f for f in group if f in parent]
    for f in members[1:]: _union(members[0], f)
comp = {}
for f in FEATS: comp.setdefault(_find(f), []).append(f)
hybrid_groups = sorted(comp.values(), key=lambda g: -len(g))

print("=== HYBRID groups (empirical + angle/hbond must-link) ===")
for g in hybrid_groups:
    print(("  GROUP:  " if len(g) > 1 else "  single:"), g)

CAT = BASE_CAT + GEOM_CAT
NUM = BASE_NUM + GEOM_NUM
FE = CAT + NUM
GROUPS_HYB = {", ".join(g): [c for c in g if c in FE] for g in hybrid_groups}

In [ ]:
# ---- per model: high/low |error| classifier + grouped permutation importance (shared groups) ----
def clf(cat, num):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
                             ("num", StandardScaler(), num)])
    return Pipeline([("pre", pre), ("m", HistGradientBoostingClassifier(
        max_depth=3, learning_rate=0.05, max_iter=400, l2_regularization=1.0,
        class_weight="balanced", random_state=0))])


imp_by_model = {}
for tag, label in MODELS.items():
    y = df[f"err{tag}"].values
    lo_thr, hi_thr = np.quantile(y, 0.50), np.quantile(y, 0.90)
    cls = np.full(len(df), -1); cls[y <= lo_thr] = 0; cls[y >= hi_thr] = 1
    sub = df[cls >= 0].copy(); Y = cls[cls >= 0]
    rng = np.random.default_rng(0)                     # same seed per model as in the separate notebooks
    fold_imp = {lab: [] for lab in GROUPS_HYB}
    base_aucs = []
    for tr, te in GroupKFold(5).split(sub[FE], Y, sub["entry"].values):
        model = clf(CAT, NUM).fit(sub[FE].iloc[tr], Y[tr])
        Xte = sub[FE].iloc[te].reset_index(drop=True); yte = Y[te]
        base_auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1]); base_aucs.append(base_auc)
        for lab, cols in GROUPS_HYB.items():
            if not cols: continue
            drops = []
            for _ in range(6):
                Xp = Xte.copy()
                Xp[cols] = Xp[cols].values[rng.permutation(len(Xp))]
                drops.append(base_auc - roc_auc_score(yte, model.predict_proba(Xp)[:, 1]))
            fold_imp[lab].append(np.mean(drops))
    impH = pd.DataFrame({"group": list(fold_imp),
                         "importance": [np.mean(v) for v in fold_imp.values()],
                         "std": [np.std(v) for v in fold_imp.values()]}).sort_values("importance")
    impH.to_csv(f"{FIG}/error_enrichment_grouped_perm_importance_hybrid{OUT_SUFFIX}_{TAG_NAME[tag]}.csv", index=False)
    imp_by_model[tag] = impH
    print(f"\n=== {label}: OOF ROC-AUC {np.mean(base_aucs):.3f} ± {np.std(base_aucs):.3f} | "
          f"thresholds low<= {lo_thr:.2f}, high>= {hi_thr:.2f} ppm ===")
    print(impH.iloc[::-1].round(4).to_string(index=False))

In [ ]:
# ---- per-model bar plot (same style as the original notebook) ----
set_style(base=16)
GROUP_LABELS = {          # key = comma-joined hybrid group; unmapped groups fall back to the raw column names
    "tdi, ss, psi_sin, psi_cos, psi_prev_sin, psi_prev_cos": "backbone conformation",
    "tdn, pack_N8":                    "local packing",
    "s2":                              "order parameter",
    "aa":                              "residue identity",
    "plddt":                           "structural reliability\n(pLDDT)",
    "p_exchange":                      "predicted μs-ms motion",
    "hb_HN_exists, hb_HN_dist, hb_HN_cosA, hb_HN_cosH, hb_HN_energy": "hydrogen bond\ngeometry",
    "phi_sin, phi_cos":                "φ angle",
    "prev_aa":                         "previous residue",
    "chi1_sin, chi1_cos":              "χ1 angle",
    "temperature":                     "temperature",
    "phi_next_sin, phi_next_cos":      "next-residue phi angle",
    "next_aa, is_prepro":              "next residue",
    "rc_N":                            "random-coil shift",
    "exposure":                        "solvent exposure",
    "ph":                              "pH",
}
unmapped = [g for g in GROUPS_HYB if g not in GROUP_LABELS]
if unmapped:
    print("groups without a GROUP_LABELS entry (shown with raw names):", unmapped)

for tag, label in MODELS.items():
    impH = imp_by_model[tag]
    impH = impH[impH["importance"] > 0.015].copy()
    impH["group"] = impH["group"].map(GROUP_LABELS).fillna(impH["group"])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(impH["group"], impH["importance"], xerr=impH["std"], color="#61AFEF", edgecolor="white")
    ax.axvline(0, color="k", lw=0.8)
    ax.axvline(0.015, color="0.5", lw=0.8, ls="--")      # display threshold: groups below 0.015 are not shown
    ax.set_xlabel("drop in AUROC when group is permuted", fontsize=16, font="Arial")
    ax.set_title(f"Ranked Importance of Feature Groups\n{label}", fontsize=16, font="Arial")
    plt.tight_layout()
    name = f"error_enrichment_grouped_perm_importance_hybrid_part{OUT_SUFFIX}_{TAG_NAME[tag]}"
    plt.savefig(f"{FIG}/{name}.svg", bbox_inches="tight")
    plt.savefig(f"{FIG}/{name}.png", dpi=300, bbox_inches="tight")
    print("saved", name); plt.show()